In [1]:

from langchain_community.document_loaders import PyPDFLoader

PDF_PATH ="random_20pages.pdf"
loader = PyPDFLoader(PDF_PATH)

pages = loader.load()

print(f"Loaded {len(pages)} pages from pdf")
print(pages[0].page_content[:100])
print("...")
print(pages[1].page_content[:100])


C:\Users\lmnav\AppData\Local\Temp\ipykernel_2028\3046935299.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
C:\Users\lmnav\OneDrive\Desktop\HEX_INTERN_PYTHON\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
incorrect startxref pointer(1)
parsing for Object Streams


Loaded 39 pages from pdf
Random Knowledge Compendium
A Collection of 20 Topics Across Science, History & Philosophy
Generated
...
Chapter 1
 The Origins of the Universe
Category: Cosmology and Space
Overview
The interplay between 


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size= 600,
    chunk_overlap = 100,
    separators=["\n\n","\n", "."," "],
)
chunks = splitter.split_documents(pages)
len(chunks)


118

In [3]:

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vector_store = Chroma.from_documents(chunks, embeddings)


'[WinError 10054] An existing connection was forcibly closed by the remote host' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

In [ ]:

retriever = vector_store.as_retriever(search_kwargs={"k":3})
test_query = "what is mean by software project management.("

retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved):
    print(f"... chunk {i}..")
    print(doc.page_content[:300])
    print(".....")

In [ ]:
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

retrieved_with_scores = vector_store.similarity_search_with_relevance_scores(test_query, k=3)

for i, (doc, score) in enumerate(retrieved_with_scores):
     print(f"... chunk {i} (Score/Relevance: {score:.4f}) ..")
     print(doc.page_content[:300])
     print(".....")    

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()     

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")


In [ ]:

SYS_PROMPT = """
Answer questions with provided context only.

Context:
{ctxt}
"""


In [ ]:
prompt =  ChatPromptTemplate.from_messages([
    ("system", SYS_PROMPT),
    ("human", "{question}")
])

chain = prompt | llm | StrOutputParser()

def ask_question(question_txt):
    docs = retriever.invoke(question_txt)
    context = "\n\n---\n\n".join(d.page_content for d in docs)
    return chain.invoke({"ctxt": context,"question": question_txt})
print("Ready")  

In [ ]:
res=ask_question("what is mean by project")
print(res)

In [ ]:
# uv add beautifulsoup4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

urls = ["https://en.wikipedia.org/wiki/Supercomputer"]
loader = WebBaseLoader(urls)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=100)
web_chunks = text_splitter.split_documents(docs)

vector_store.add_documents(web_chunks)
print(f"Added {len(web_chunks)} web chunks to Chroma.")

ask_question("who invented super computer?")

In [ ]:
# uv add beautifulsoup4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

urls = ["https://en.wikipedia.org/wiki/Seymour_Cray"]
loader = WebBaseLoader(urls)
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=100)
web_chunks = text_splitter.split_documents(docs)

vector_store.add_documents(web_chunks)
print(f"Added {len(web_chunks)} web chunks to Chroma.")

ask_question("who invented super computer?")

In [ ]:
# pydantic
from typing import List
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI


In [ ]:
# 1. Define the desired JSON structure using Pydantic
class userinfo(BaseModel):
    name: str | None = Field(description="The name of the person.")
    age: str | None = Field(description="age of the person")
    

In [ ]:
pydantic_parser = PydanticOutputParser(pydantic_object=userinfo)

In [ ]:
SYS_PROMPT = """
Answer questions with provided context only.
{format_instructions}

Context:
{ctxt}
"""

prompt =  ChatPromptTemplate.from_messages([
    ("system", SYS_PROMPT),
    ("human", "{question}")
])


In [ ]:
def ask_questionss(question_txt):
    docs = retriever.invoke(question_txt)
    context = "\n\n---\n\n".join(d.page_content for d in docs)
    # print(f"....{context}....")
    # print(f"...pydantic format inst:{ pydantic_parser.get_format_instructions()}....")
    return chain.invoke({"ctxt": context,"question": question_txt, "format_instructions": pydantic_parser.get_format_instructions()}) 
    # return chain.invoke({"ctxt": context,"question": question_txt, "format_instructions": "provide only name and age in json"}) 
     
print("Ready")

res= ask_questionss("who is father of computer?")
print("------")
print(res)